# 19 — MIDI 2026: Abstention & Truncation Diagnostics (v1)

**Hakem 2'nin iki talebi:** (i) Somut cevaplar ile açıkça cevap vermekten kaçınan çıktıları ayrı incelemek, (ii) 20-token sınırının neden olduğu olası kesilmeleri sayısallaştırmak.

- **Yalnızca mevcut kayıtlı Stage 08/14/16/17/18 dosyaları okunur.** LLM çağrısı, model eğitimi veya bootstrap yoktur.
- Test soruları Stage 14'te dondurulmuş 250 soru ID'sinden alınır. Development veya başka test soruları eklenmez.
- `unknown_response` Stage 08'deki **pure unknown** göstergesidir. `truncated_by_max_tokens` orijinal kayıt pipeline'ının kesilme **bayrağıdır**; yalnızca 20 tokena ulaşmakla eşanlamlı değildir.
- `sample_truncation_rate`, her soru için beş tutarlılık örneklemesinin *ayrı* truncation oranını temsil eder; primer cevapla karıştırılmaz.
- `master_feature_table_v1.parquet` mevcutsa (Stage 08 manifestindeki hash doğrulandıktan sonra), `empty_response`, `ended_with_eos` ve üretim süreleri de kullanılır; yoksa üçlü sınıflandırma **candidate_substantive** olarak işaretlenir.
- ECE ve AUROC küçük ya da tek sınıflı alt gruplarda yanıltıcı/hesaplanamaz olabilir; hesaplanamayanlar boş hücre olarak kalır.
- **Araştırma statüsü:** `post_review_exploratory`. Önceden sabitlenen Stage 12–14 birincil sonuçları değiştirilmez.
- Yalnızca `results/midi2026_revision/v1/03_response_diagnostics/` ve `manifests/` içerisine, üzerine yazmadan kayıt yapılır.

**Çalıştırma:** Colab'da bütün kod hücrelerini sırayla yürütün; `INPUT_MODE='drive'` varsayılandır.

In [ ]:
# 1 — Ortam ve dosya yolları (hiçbir LLM modeli yüklenmez)
import sys, subprocess, importlib.util
if importlib.util.find_spec('pyarrow') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','pyarrow>=15,<25'])

from pathlib import Path
import os, json, hashlib, math, platform
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

INPUT_MODE='drive'   # 'drive' (Colab) veya 'local'
if INPUT_MODE=='drive':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
elif INPUT_MODE=='local':
    ROOT=Path('/content/LLM_Confidence_Thesis')
else:
    raise ValueError('INPUT_MODE drive/local olmalı')
assert ROOT.is_dir(), f'Proje kök klasörü bulunamadı: {ROOT}'

DEV='validated_v007'; TEST='validated_v012_final250_v1'
REV=ROOT/'results'/'midi2026_revision'/'v1'
OUT=REV/'03_response_diagnostics'; MAN=REV/'manifests'
KEY=['model_id','question_id']; TARGET='manual_is_correct'
CORE_METHODS=['raw_verification','logistic_core','xgb_core']
ADDITIONAL_METHODS=['platt_verification','isotonic_verification']

P={
 'test_extended':ROOT/'features'/'assembled'/TEST/'extended_model_matrix_v1.parquet',
 'test_manifest':ROOT/'features'/'assembled'/TEST/'stage08_manifest_v1.json',
 'test_master':ROOT/'features'/'assembled'/TEST/'master_feature_table_v1.parquet',
 'frozen_predictions':ROOT/'results'/'stage14'/TEST/'final_test_predictions_v1.parquet',
 'pred_manifest':ROOT/'results'/'stage14'/TEST/'stage14_predictions_manifest_v1.json',
 'stage16_summary':REV/'00_integrity_audit'/'audit_summary_v1.json',
 'stage17_predictions':REV/'01_self_verification_baseline'/'stage17_single_signal_predictions_v1.parquet',
 'stage17_pred_manifest':MAN/'stage17_prediction_manifest_v1.json',
 'stage17_eval_manifest':MAN/'stage17_evaluation_manifest_v1.json',
 'stage18_eval_manifest':MAN/'stage18_evaluation_manifest_v2.json',
}
O={
 'counts':OUT/'stage19_response_group_counts_v1.csv',
 'subgroup_metrics':OUT/'stage19_subgroup_metrics_v1.csv',
 'truncation':OUT/'stage19_truncation_summary_v1.csv',
 'cross_tab':OUT/'stage19_abstention_truncation_cross_tab_v1.csv',
 'flagged_rows':OUT/'stage19_flagged_response_ids_v1.csv',
 'report':OUT/'stage19_run_report_v1.md',
 'manifest':MAN/'stage19_evaluation_manifest_v1.json',
}
print('ROOT:',ROOT,'\nOUTPUT:',OUT)

In [ ]:
# 2 — Strict input integrity, immutable writes

def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def digest(path):
    p=Path(path);require(p.is_file(),f'Girdi eksik: {p}')
    h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()

def readjson(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def immutable_write(path,data):
    path=Path(path)
    require(path.parent in (OUT,MAN),'Sadece revision klasörüne yazılabilir')
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        require(path.read_bytes()==data,f'Çıktı değişikliği saptandı: {path} (v2 olarak yeniden oluşturun)')
        print('UNCHANGED:',path.name)
    else:
        with path.open('xb') as f:f.write(data)
        print('CREATED:',path.name)
    return digest(path)

def save_csv(path,frame):
    blob=frame.to_csv(index=False,na_rep='',float_format='%.15g',lineterminator='\n').encode('utf-8')
    return immutable_write(path,blob)

def save_json(path,obj):
    blob=(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n').encode('utf-8')
    return immutable_write(path,blob)

REQUIRED=['test_extended','test_manifest','frozen_predictions','pred_manifest','stage16_summary','stage18_eval_manifest']
for name in REQUIRED:require(P[name].is_file(),f'Gerekli dosya yok: {name}: {P[name]}')
s16=readjson(P['stage16_summary']);s18=readjson(P['stage18_eval_manifest'])
manifest08=readjson(P['test_manifest']);manifest14=readjson(P['pred_manifest'])
require(s16.get('status')=='PASS' and not s16.get('required_unresolved_checks'),'Stage 16 PASS değil')
require(s18.get('status')=='PASS' and s18.get('study_status')=='post_review_exploratory','Stage 18 PASS değil')
require(manifest14.get('status')=='PREDICTIONS_FROZEN_BEFORE_TEST_LABEL_EVALUATION','Stage14 freeze statüsü')
require(s18.get('stage14_predictions_sha256')==digest(P['frozen_predictions']),'Stage18 frozen source SHA')
require(s18.get('stage16_summary_sha256')==digest(P['stage16_summary']),'Stage18 Stage16 bağlantısı')
require(manifest08['outputs']['extended_model_matrix_sha256']==digest(P['test_extended']),'Extended matris SHA yanlış')
require(manifest14['predictions_sha256']==digest(P['frozen_predictions']),'Stage14 frozen prediction SHA yanlış')
require(s16['input_sha256']['test_extended']==digest(P['test_extended']),'Stage16 sonrası test features değişmiş')
require(s16['input_sha256']['frozen_predictions']==digest(P['frozen_predictions']),'Stage16 sonrası frozen preds değişmiş')

HAVE_MASTER=P['test_master'].is_file()
if HAVE_MASTER:
    require(digest(P['test_master'])==manifest08['outputs']['master_feature_table_sha256'],'Stage08 master SHA mismatch')
    print('Optional master: VERIFIED — EOS, empty-response, duration available')
else:
    print('Optional master: NOT FOUND — substantive candidate sınıflaması sınırlı yapılacak')

HAVE_STAGE17=P['stage17_predictions'].is_file()
if HAVE_STAGE17:
    require(P['stage17_pred_manifest'].is_file() and P['stage17_eval_manifest'].is_file(), 'Stage17 dosyaları kısmi')
    pm17=readjson(P['stage17_pred_manifest']);em17=readjson(P['stage17_eval_manifest'])
    require(pm17.get('predictions_sha256')==digest(P['stage17_predictions']),'Stage17 tahmin SHA yanlış')
    require(em17.get('status')=='PASS' and em17.get('source_stage14_predictions_sha256')==digest(P['frozen_predictions']),'Stage17 evaluation manifest yanlış')
print('PROVENANCE PASS | HAVE_MASTER:',HAVE_MASTER,'| HAVE_STAGE17:',HAVE_STAGE17)

In [ ]:
# 3 — Aynı 250 final-test sorusunun bayraklarını al; önce cevap türlerini sınıflandır
BASE_COLS=KEY+[TARGET,'unknown_response','contains_unknown_marker',
          'mixed_answer_unknown','refusal_indicator','truncated_by_max_tokens',
          'sample_truncation_rate','output_content_token_count','self_consistency']
base_all=pd.read_parquet(P['test_extended'],columns=BASE_COLS)
require(len(base_all)==3750 and not base_all.duplicated(KEY).any(),'Test extended 3750/key sözleşmesi')
frozen=pd.read_parquet(P['frozen_predictions'],columns=KEY+['method','p_correct'])
require(len(frozen)==3000 and not frozen.duplicated(KEY+['method']).any(),'Stage14 frozen predictions')
require(set(frozen['method'])=={'raw_verification','train_prevalence','logistic_core','xgb_core'},'Stage14 yöntemleri')
selected_ids=set(frozen['question_id'].astype(str))
require(len(selected_ids)==250,'Seçilen test sorusu sayısı 250 değil')
base=base_all.loc[base_all['question_id'].astype(str).isin(selected_ids)].copy()
require(len(base)==750 and base.groupby('model_id').size().eq(250).all(),'Test 750 cevap değil')
require(base[TARGET].notna().all() and base[TARGET].isin([0,1]).all(),'Etiketler 0/1 değil')
for c in ['unknown_response','contains_unknown_marker','mixed_answer_unknown','refusal_indicator','truncated_by_max_tokens']:
    require(base[c].notna().all() and base[c].isin([0,1,False,True]).all(),f'{c} ikili değil')
require(base['sample_truncation_rate'].between(0,1).all(),'Sample truncation rate 0..1 dışı')
require(np.isfinite(base['output_content_token_count'].to_numpy(dtype=float)).all(),'Output length NaN')

if HAVE_MASTER:
    master_cols=KEY+['empty_response','ended_with_eos','generation_seconds']
    master=pd.read_parquet(P['test_master'],columns=master_cols)
    require(len(master)==3750 and not master.duplicated(KEY).any(),'Master 3750/key')
    base=base.merge(master,on=KEY,how='left',validate='one_to_one',indicator=True)
    require(base['_merge'].eq('both').all(),'Master test pair eksiği')
    base=base.drop(columns=['_merge'])
    for c in ['empty_response','ended_with_eos']:
        require(base[c].notna().all(),f'Master {c} missing')
else:
    base['empty_response']=pd.Series([pd.NA]*len(base),dtype='boolean',index=base.index)
    base['ended_with_eos']=pd.Series([pd.NA]*len(base),dtype='boolean',index=base.index)
    base['generation_seconds']=np.nan

explicit=base['unknown_response'].astype(bool)
other=(base['contains_unknown_marker'].astype(bool)|
       base['mixed_answer_unknown'].astype(bool)|
       base['refusal_indicator'].astype(bool)) & (~explicit)
if HAVE_MASTER:
    other=other| (base['empty_response'].astype(bool)&(~explicit))
base['abstention_group']=np.where(explicit,'explicit_abstention','non_explicit_abstention')
base['response_group']=np.select([explicit,other],['explicit_abstention','other_ambiguous_or_nonanswer'],
                                 default=('substantive' if HAVE_MASTER else 'candidate_substantive'))
base['primary_truncation_group']=np.where(base['truncated_by_max_tokens'].astype(bool),
                                           'flagged_truncated','not_flagged_truncated')
base['any_sample_flagged_truncated']=base['sample_truncation_rate'].astype(float)>0

# Conditional evaluations always include the frozen original methods; Stage17 if available.
preds=frozen.loc[frozen['method'].isin(CORE_METHODS)].copy()
if HAVE_STAGE17:
    add=pd.read_parquet(P['stage17_predictions'],columns=KEY+['method','p_correct'])
    require(len(add)==1500 and set(add['method'])==set(ADDITIONAL_METHODS),'Stage17 2 method x 750')
    preds=pd.concat([preds,add],ignore_index=True)
require(not preds.duplicated(KEY+['method']).any(),'Method prediction key duplicate')
require(preds['p_correct'].between(0,1).all(),'Confidence aralığı [0,1] dışı')
require(set(map(tuple,preds[KEY].drop_duplicates().to_numpy()))==set(map(tuple,base[KEY].to_numpy())),
        'Tahmin cevap ID eşleşmesi bozuk')
joined=preds.merge(base,on=KEY,how='inner',validate='many_to_one')
require(len(joined)==750*(len(CORE_METHODS)+(len(ADDITIONAL_METHODS) if HAVE_STAGE17 else 0)),
        'Joined predictions satır sayısı yanlış')
print('FINAL TEST ANSWERS:',len(base),'| EVALUATION ROWS:',len(joined))
print(pd.crosstab(base['model_id'],base['response_group']).to_string())
print(pd.crosstab(base['model_id'],base['primary_truncation_group']).to_string())

In [ ]:
# 4 — Cevap grupları: sayı, cevap doğruluğu ve bayraklar (model-bazlı)
def wilson_bounds(success,n,z=1.95996398454):
    if n==0:return (float('nan'),float('nan'))
    ph=success/n;den=1+z*z/n
    center=(ph+z*z/(2*n))/den
    half=z*math.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return max(0.,center-half),min(1.,center+half)

counts=[]
for mid,frame in base.groupby('model_id',sort=True):
    for grouping in ['abstention_group','response_group','primary_truncation_group']:
        for group_name,sub in frame.groupby(grouping,sort=True):
            n=len(sub);pos=int(sub[TARGET].sum());low,high=wilson_bounds(pos,n)
            counts.append({'model_id':mid,'grouping':grouping,'group':group_name,'n':n,
                'proportion_of_model':n/len(frame),'correct_n':pos,'incorrect_n':n-pos,
                'correctness_rate':pos/n,'correctness_wilson95_low':low,'correctness_wilson95_high':high,
                'primary_truncation_rate':float(sub['truncated_by_max_tokens'].astype(int).mean()),
                'mean_content_tokens':float(sub['output_content_token_count'].mean()),
                'mean_sample_truncation_rate':float(sub['sample_truncation_rate'].mean())})
counts_df=pd.DataFrame(counts).sort_values(['model_id','grouping','group']).reset_index(drop=True)
print(counts_df[['model_id','grouping','group','n','correctness_rate']].to_string(index=False))

In [ ]:
# 5 — Her model, yöntem ve cevap/truncation alt grubunda kalibrasyon metrikleri
#   Küçük gruplarda ve tek sınıflı alt gruplarda AUROC = NA; ECE yorumu ihtiyatlı.
def eval_metrics(df,bins=10):
    y=df[TARGET].to_numpy(dtype=int);p=df['p_correct'].to_numpy(dtype=float)
    n=len(y)
    if not n:
        return {'n':0,'correct_n':0,'correctness_rate':np.nan,'mean_confidence':np.nan,
                'confidence_gap':np.nan,'brier':np.nan,'ece_10bins':np.nan,'auroc':np.nan}
    bc=np.minimum((np.clip(p,0,1)*bins).astype(int),bins-1)
    ece=sum((bc==j).mean()*abs(y[bc==j].mean()-p[bc==j].mean()) for j in range(bins) if np.any(bc==j))
    return {'n':int(n),'correct_n':int(y.sum()),'correctness_rate':float(y.mean()),
      'mean_confidence':float(p.mean()),'confidence_gap':float(p.mean()-y.mean()),
      'brier':float(np.mean((p-y)**2)), 'ece_10bins':float(ece),
      'auroc':float(roc_auc_score(y,p)) if np.unique(y).size==2 else np.nan}

subgroup_rows=[]
for (mid,method),frame in joined.groupby(['model_id','method'],sort=True):
    # Bir yöntem/model için koşulsuz özet de raporlanır (Stage 14 ile kontrol için)
    subgroup_rows.append({'model_id':mid,'method':method,'grouping':'all','group':'all',**eval_metrics(frame)})
    for group_col in ['abstention_group','response_group','primary_truncation_group']:
        for name,part in frame.groupby(group_col,sort=True):
            subgroup_rows.append({'model_id':mid,'method':method,'grouping':group_col,'group':name,
                                  **eval_metrics(part)})
metrics_df=pd.DataFrame(subgroup_rows).sort_values(['model_id','method','grouping','group']).reset_index(drop=True)
require(metrics_df.query('grouping=="all"')['n'].eq(250).all(),'250 soruluk yöntem alt kümesi yanlış')
print(metrics_df.query('grouping=="abstention_group" and method in ["raw_verification","logistic_core","xgb_core"]')
      [['model_id','method','group','n','correctness_rate','mean_confidence','brier']].to_string(index=False))

In [ ]:
# 6 — Primer 20-token truncation bayrağı ve K=5 resampling kesilmelerini ayrı raporla
truncation_rows=[];cross=[]
for mid,group in base.groupby('model_id',sort=True):
    n=len(group);prim=group['truncated_by_max_tokens'].astype(int)
    smp=group['sample_truncation_rate'].astype(float)
    eos=group['ended_with_eos'] if HAVE_MASTER else None
    # Her soru için K=5 ayrı örneğin ortalama truncation oranı: 0,0.2,...,1.0 olmalı.
    frac_error=np.abs(smp.to_numpy()*5-np.round(smp.to_numpy()*5))
    require(np.all(frac_error<1e-5),'K=5 sample_truncation_rate 1/5 ile uyumlu değil')
    tr=group.loc[prim.eq(1)];non=group.loc[prim.eq(0)]
    low,high=wilson_bounds(int(prim.sum()),n)
    truncation_rows.append({
      'model_id':mid,'n_test_answers':n,'max_new_content_tokens_configured':20,
      'primary_flagged_truncated_n':int(prim.sum()),'primary_flagged_truncated_rate':float(prim.mean()),
      'primary_flag_wilson95_low':low,'primary_flag_wilson95_high':high,
      'primary_unflagged_n':int((1-prim).sum()),
      'mean_primary_content_tokens':float(group['output_content_token_count'].mean()),
      'content_token_count_ge20_n':int((group['output_content_token_count']>=20).sum()),
      'primary_flagged_correctness_rate':float(tr[TARGET].mean()) if len(tr) else np.nan,
      'primary_unflagged_correctness_rate':float(non[TARGET].mean()) if len(non) else np.nan,
      'mean_sample_truncation_rate_K5':float(smp.mean()),
      'sample_any_flagged_n':int((smp>0).sum()),
      'estimated_sample_flagged_total_out_of_5n':int(round((smp*5).sum())),
      'estimated_sample_total':int(5*n),
      'ended_with_eos_n':int(eos.astype(int).sum()) if HAVE_MASTER else np.nan,
      'ended_with_eos_available':bool(HAVE_MASTER),
      'mean_primary_generation_seconds':float(group['generation_seconds'].mean()) if HAVE_MASTER else np.nan,
    })
    tab=pd.crosstab(group['abstention_group'],group['primary_truncation_group'])
    for a in ['explicit_abstention','non_explicit_abstention']:
        for t in ['flagged_truncated','not_flagged_truncated']:
            cross.append({'model_id':mid,'abstention_group':a,'truncation_group':t,
                          'n':int(tab.loc[a,t]) if a in tab.index and t in tab.columns else 0})
trunc_df=pd.DataFrame(truncation_rows).sort_values('model_id').reset_index(drop=True)
cross_df=pd.DataFrame(cross).sort_values(['model_id','abstention_group','truncation_group']).reset_index(drop=True)
print(trunc_df.to_string(index=False))

In [ ]:
# 7 — Kimlik düzeyinde tanı koymak için yalnız bayraklı soruları raporla
# Soru metni / referans cevap / ham model cevabı bu matrislerde yoktur;
# bu notebook bu bilgileri uydurmaz. Gerçek örnekler Stage 21'de ilgili ham dosyadan getirilecek.
flag=(base['unknown_response'].astype(bool)|base['truncated_by_max_tokens'].astype(bool)|
      base['sample_truncation_rate'].astype(float).gt(0)|
      base['refusal_indicator'].astype(bool)|base['mixed_answer_unknown'].astype(bool))
flagged=base.loc[flag,KEY+['response_group','abstention_group','primary_truncation_group',
                          'unknown_response','refusal_indicator','mixed_answer_unknown',
                          'output_content_token_count','sample_truncation_rate',TARGET]].copy()
flagged=flagged.sort_values(KEY).reset_index(drop=True)
print('Flagged model-answers:',len(flagged),'of',len(base))

In [ ]:
# 8 — Immutable çıktı dosyaları; rapor ve manifest
out_sha={}
for key,frame in [('counts',counts_df),('subgroup_metrics',metrics_df),('truncation',trunc_df),
                  ('cross_tab',cross_df),('flagged_rows',flagged)]:
    out_sha[key]=save_csv(O[key],frame)

model_labels={
 'Qwen/Qwen3-4B':'Qwen3-4B',
 'google/gemma-3-4b-it':'Gemma-3-4B-it',
 'microsoft/Phi-4-mini-instruct':'Phi-4-mini-instruct'}
lines=['# MIDI2026 Stage 19 — Abstention & Truncation Diagnostics','',
 '**Status:** PASS','',
 '**Nature:** post-review exploratory; no new LLM inference, calibration or bootstrap.','',
 '- Independent selected final test: 250 questions per model (750 model–answers).',
 '- Primary truncation numbers use Stage08 `truncated_by_max_tokens` flag.',
 '- Five-sample truncation numbers use distinct `sample_truncation_rate` (K=5).',
 '- `unknown_response` is explicit pure-unknown marker, not any possible abstention.',
 '- The reference correctness label for an abstention is taken as saved, not reinterpreted.',
 '- AUROC undefined for groups with a single label; small-subgroup metrics are descriptive.',
 f'- `master_feature_table` verified: {HAVE_MASTER}.',
 f'- Stage17 additional single-feature calibrator predictions included: {HAVE_STAGE17}.','',
 '## Response-group and truncation overview','',
 '| Model | Explicit unknown | Other ambiguous / nonanswer | Primary truncated flag | Sample truncation rate (K=5) |',
 '|---|---:|---:|---:|---:|']
for mid in sorted(base['model_id'].unique()):
    b=base.loc[base.model_id.eq(mid)];t=trunc_df.loc[trunc_df.model_id.eq(mid)].iloc[0]
    lines.append(f"| {model_labels.get(mid,mid)} | {int(b.unknown_response.sum())}/250 | "+
      f"{int(b.response_group.eq('other_ambiguous_or_nonanswer').sum())}/250 | "+
      f"{int(t.primary_flagged_truncated_n)}/250 | {t.mean_sample_truncation_rate_K5:.3f} |")
lines += ['', '## Notes for the paper', '',
 '- Do not equate hitting the 20-token count with confirmed truncation; use the recorded flag and EOS (if available).',
 '- Non-explicit abstentions and candidate substantive outputs should not be silently conflated with verified substantive answers.',
 '- Models differ in abstention rates and some strata may be too small for reliable AUROC/ECE inference.',
 '- Reporting Brier/accuracy within each stratum does **not** establish a causal effect of abstention or truncation.',
 '- All metrics reflect confidence estimation; post-hoc calibration does not change the underlying answers.',
 '- Full prompt templates and raw response text require earlier Stage04–07 artifacts, outside this notebook.',
 '', '## Generated files','']
lines += [f'- `{O[k].name}`' for k in ['counts','subgroup_metrics','truncation','cross_tab','flagged_rows']]
lines += ['- `stage19_evaluation_manifest_v1.json`','']
report='\n'.join(lines)
out_sha['report']=immutable_write(O['report'],report.encode('utf-8'))
inputs={
 'stage08_test_extended':digest(P['test_extended']),
 'stage08_test_manifest':digest(P['test_manifest']),
 'stage14_frozen_predictions':digest(P['frozen_predictions']),
 'stage14_predictions_manifest':digest(P['pred_manifest']),
 'stage16_summary':digest(P['stage16_summary']),
 'stage18_evaluation_manifest':digest(P['stage18_eval_manifest']),
}
if HAVE_MASTER:inputs['stage08_test_master']=digest(P['test_master'])
if HAVE_STAGE17:
    inputs['stage17_single_signal_predictions']=digest(P['stage17_predictions'])
    inputs['stage17_prediction_manifest']=digest(P['stage17_pred_manifest'])
manifest={
 'stage':'19_midi2026_abstention_truncation_diagnostics',
 'version':'v1','status':'PASS','study_status':'post_review_exploratory',
 'n_models':3,'n_test_questions_per_model':250,'n_model_answer_rows':len(base),
 'method_names':sorted(joined['method'].unique().tolist()),
 'n_method_answer_rows':len(joined),'llm_calls':0,'calibrator_fits':0,'bootstrap_repeats':0,
 'abstention_definition':'unknown_response==1 (Stage08 pure-unknown flag)',
 'response_group_substantive_certainty':('empty/refusal/unknown markers excluded' if HAVE_MASTER else 'candidate only: raw empty-response flag not available'),
 'primary_truncation_definition':'Stage08 truncated_by_max_tokens flag; not implied by token count alone',
 'resample_truncation_definition':'Stage08 mean sample_truncation_rate over K=5 resamples',
 'has_verified_stage08_master':HAVE_MASTER,
 'original_stage14_modified':False,
 'input_sha256':inputs,'outputs_sha256':out_sha,
 'interpretation_notes':['Post-review exploratory descriptive analyses, not preregistered',
     'Correctness labels unchanged','No test labels used for fitting','Small or one-class subgroup scores are descriptive'],
}
manifest_sha=save_json(O['manifest'],manifest)
print('\nSTAGE 19 PASS — manifest SHA:',manifest_sha)
print('Output directory:',OUT)
print(report)

## Bana gönderilecek çıktılar

Notebook tamamlandığında şu dosyaları gönder:

1. `stage19_response_group_counts_v1.csv`
2. `stage19_subgroup_metrics_v1.csv`
3. `stage19_truncation_summary_v1.csv`
4. `stage19_abstention_truncation_cross_tab_v1.csv`
5. `stage19_run_report_v1.md`
6. `stage19_evaluation_manifest_v1.json`

`stage19_flagged_response_ids_v1.csv` isteğe bağlıdır; soru kimliklerinin paylaşılması uygunsa ayrıca incelenebilir.

**Makale notu:** Ham soru/model cevabı metinleri bu notebook'un girdilerinde yoktur. Hakemin istediği gerçek soru/cevap/güven örnekleri için Stage 04–07 kayıtlarına, doğrulama metodolojisi için veri seti oluşturma kayıtlarına ayrıca bakılacaktır.